# Lab 3: Data Preprocessing & Feature Engineering

This lab focuses on cleaning and preparing messy data for machine learning models. We will handle missing values, encode categorical variables, scale features, and evaluate models.


## Task 1: Exploring the Messy Data

In this task, we explore the dataset to understand its structure and identify issues such as missing values, data types, and categorical distributions.


In [49]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200

data = {
    'age': np.random.randint(18, 80, n).astype(float),
    'blood_pressure': np.round(np.random.uniform(90, 180, n), 1),
    'cholesterol': np.round(np.random.uniform(150, 350, n), 1),
    'bmi': np.round(np.random.uniform(18, 42, n), 1),
    'gender': np.random.choice(['Male', 'Female'], n),
    'city': np.random.choice(['Jeddah', 'Riyadh', 'Dammam', 'Makkah'], n),
    'smoker': np.random.choice(['Yes', 'No'], n, p=[0.3, 0.7]),
    'heart_disease': np.random.choice([0, 1], n, p=[0.6, 0.4])
}

df = pd.DataFrame(data)

# Add missing values
missing_idx = np.random.choice(n, 20, replace=False)
df.loc[missing_idx[:10], 'age'] = np.nan
df.loc[missing_idx[10:15], 'blood_pressure'] = np.nan
df.loc[missing_idx[15:], 'cholesterol'] = np.nan

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nMissing values:")
print(df.isnull().sum())


Dataset shape: (200, 8)

First 5 rows:
    age  blood_pressure  cholesterol   bmi  gender    city smoker  \
0  56.0           171.7        295.2  34.4    Male  Riyadh     No   
1  69.0           112.4        345.2  21.9  Female  Dammam     No   
2  46.0           126.9        253.3  39.9  Female  Jeddah     No   
3  32.0           158.0        214.6  37.7  Female  Riyadh     No   
4  60.0           110.6        309.0  40.8    Male  Makkah    Yes   

   heart_disease  
0              1  
1              0  
2              0  
3              1  
4              1  

Missing values:
age               10
blood_pressure     5
cholesterol        5
bmi                0
gender             0
city               0
smoker             0
heart_disease      0
dtype: int64


In [50]:
print(df.info())

print("\nMissing % per column:")
print((df.isnull().sum() / len(df) * 100).round(2))

print("\nCity counts:")
print(df['city'].value_counts())

print("\nColumn with most missing values:")
print(df.isnull().sum().idxmax())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             190 non-null    float64
 1   blood_pressure  195 non-null    float64
 2   cholesterol     195 non-null    float64
 3   bmi             200 non-null    float64
 4   gender          200 non-null    object 
 5   city            200 non-null    object 
 6   smoker          200 non-null    object 
 7   heart_disease   200 non-null    int64  
dtypes: float64(4), int64(1), object(3)
memory usage: 12.6+ KB
None

Missing % per column:
age               5.0
blood_pressure    2.5
cholesterol       2.5
bmi               0.0
gender            0.0
city              0.0
smoker            0.0
heart_disease     0.0
dtype: float64

City counts:
city
Dammam    59
Jeddah    52
Riyadh    47
Makkah    42
Name: count, dtype: int64

Column with most missing values:
age


### Task 1 Conclusion

From the analysis:
- The column with the most missing values is **age (5%)**
- Blood pressure and cholesterol also have missing values (2.5%)
- BMI, gender, city, smoker, and heart_disease have no missing values

This shows that the dataset contains missing data that must be handled before training machine learning models.


## Task 2: Handling Missing Values

In this task, we experiment with different strategies for handling missing data.

We will:
- Use mean imputation instead of median
- Compare results
- Try dropping rows with missing values
- Evaluate the impact of each method


In [51]:
from sklearn.impute import SimpleImputer

# Recreate dataset (IMPORTANT - fresh start)
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200

data = {
    'age': np.random.randint(18, 80, n).astype(float),
    'blood_pressure': np.round(np.random.uniform(90, 180, n), 1),
    'cholesterol': np.round(np.random.uniform(150, 350, n), 1),
    'bmi': np.round(np.random.uniform(18, 42, n), 1),
    'gender': np.random.choice(['Male', 'Female'], n),
    'city': np.random.choice(['Jeddah', 'Riyadh', 'Dammam', 'Makkah'], n),
    'smoker': np.random.choice(['Yes', 'No'], n, p=[0.3, 0.7]),
    'heart_disease': np.random.choice([0, 1], n, p=[0.6, 0.4])
}

df = pd.DataFrame(data)

missing_idx = np.random.choice(n, 20, replace=False)
df.loc[missing_idx[:10], 'age'] = np.nan
df.loc[missing_idx[10:15], 'blood_pressure'] = np.nan
df.loc[missing_idx[15:], 'cholesterol'] = np.nan

# Apply MEAN imputation
numeric_cols = ['age', 'blood_pressure', 'cholesterol']

imputer_mean = SimpleImputer(strategy='mean')
df[numeric_cols] = imputer_mean.fit_transform(df[numeric_cols])

print("After mean imputation:")
print(df[numeric_cols].head())

# Drop rows method
df_dropped = df.dropna()

print("\nOriginal shape:", df.shape)
print("After dropping rows:", df_dropped.shape)
print("Rows lost:", len(df) - len(df_dropped))


After mean imputation:
    age  blood_pressure  cholesterol
0  56.0           171.7        295.2
1  69.0           112.4        345.2
2  46.0           126.9        253.3
3  32.0           158.0        214.6
4  60.0           110.6        309.0

Original shape: (200, 8)
After dropping rows: (200, 8)
Rows lost: 0


### Task 2 Conclusion

Using mean imputation successfully filled all missing values without reducing the dataset size.

When applying the dropna() method after imputation, no rows were removed because there were no remaining missing values. However, if dropna() were applied before imputation, it would reduce the dataset size and potentially lead to loss of important information.

Therefore, imputation is generally preferred over dropping rows when dealing with missing values in this dataset.


## Task 3: Encoding Categorical Variables

In this task, we convert categorical variables into numerical format so they can be used in machine learning models.

We will:
- Apply Label Encoding for binary variables
- Apply One-Hot Encoding for multi-category variables
- Examine the resulting dataset


In [52]:
from sklearn.preprocessing import LabelEncoder

# Label Encoding (binary columns)
le = LabelEncoder()

df['gender_encoded'] = le.fit_transform(df['gender'])
df['smoker_encoded'] = le.fit_transform(df['smoker'])

# One-Hot Encoding (multi-category column)
city_dummies = pd.get_dummies(df['city'], prefix='city')

# Combine and drop original columns
df = pd.concat([df, city_dummies], axis=1)
df = df.drop(columns=['gender', 'city', 'smoker'])

# Show results
print(df.head(10))
print("\nNumber of columns:", df.shape[1])


    age  blood_pressure  cholesterol   bmi  heart_disease  gender_encoded  \
0  56.0           171.7        295.2  34.4              1               1   
1  69.0           112.4        345.2  21.9              0               0   
2  46.0           126.9        253.3  39.9              0               0   
3  32.0           158.0        214.6  37.7              1               0   
4  60.0           110.6        309.0  40.8              1               1   
5  25.0            96.9        204.2  35.4              0               0   
6  78.0           116.1        237.8  32.7              1               0   
7  38.0           104.5        165.7  28.0              0               0   
8  56.0           173.7        155.1  40.4              0               1   
9  75.0           162.7        342.5  38.8              0               1   

   smoker_encoded  city_Dammam  city_Jeddah  city_Makkah  city_Riyadh  
0               0        False        False        False         True  
1       

### Task 3 Conclusion

Label Encoding was used for binary variables such as gender and smoker, converting them into numerical values (0 and 1).

One-Hot Encoding was applied to the city column, creating separate binary columns for each city. This avoids introducing any false order between categories.

The number of features increased after encoding, which is expected when using One-Hot Encoding. While this improves model understanding of categorical data, too many categories could increase dimensionality and affect performance.

If a feature had many categories (e.g., 50), One-Hot Encoding would create many new columns, which could increase computational cost and potentially lead to overfitting.


## Task 4: Feature Scaling

In this task, we apply feature scaling and compare different scaling methods.

We will:
- Apply MinMaxScaler
- Compare it with StandardScaler
- Observe how scaling affects feature values


In [53]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Separate features and target
X = df.drop(columns=['heart_disease']).values
y = df['heart_disease'].values

# Apply StandardScaler
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(X)

# Apply MinMaxScaler
scaler_mm = MinMaxScaler()
X_mm = scaler_mm.fit_transform(X)

# Print results
print("StandardScaler:")
print("Mean:", X_std.mean().round(2))
print("Std:", X_std.std().round(2))

print("\nMinMaxScaler:")
print("Min:", X_mm.min().round(2))
print("Max:", X_mm.max().round(2))


StandardScaler:
Mean: -0.0
Std: 1.0

MinMaxScaler:
Min: 0.0
Max: 1.0


### Task 4 Conclusion

StandardScaler transforms data so that it has a mean of 0 and a standard deviation of 1, which is useful for many machine learning algorithms.

MinMaxScaler scales data to a range between 0 and 1, preserving the shape of the distribution but making all values comparable.

MinMaxScaler is more sensitive to outliers, while StandardScaler is generally more robust. Therefore, the choice of scaler depends on the dataset and the model being used.


## Task 5: Model Comparison

In this task, we compare the performance of KNN and Decision Tree models with and without feature scaling.

We will:
- Train KNN before and after scaling
- Train Decision Tree before and after scaling
- Compare accuracy results
- Analyze the impact of scaling


In [54]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# -------- WITHOUT SCALING --------
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("WITHOUT SCALING")
print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))

# -------- WITH SCALING --------
# Use MinMaxScaler version
X_train_mm, X_test_mm, y_train_mm, y_test_mm = train_test_split(X_mm, y, test_size=0.2, random_state=42)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_mm, y_train_mm)
y_pred_knn_scaled = knn_scaled.predict(X_test_mm)

dt_scaled = DecisionTreeClassifier(random_state=42)
dt_scaled.fit(X_train_mm, y_train_mm)
y_pred_dt_scaled = dt_scaled.predict(X_test_mm)

print("\nWITH SCALING")
print("KNN Accuracy:", accuracy_score(y_test_mm, y_pred_knn_scaled))
print("Decision Tree Accuracy:", accuracy_score(y_test_mm, y_pred_dt_scaled))


WITHOUT SCALING
KNN Accuracy: 0.55
Decision Tree Accuracy: 0.575

WITH SCALING
KNN Accuracy: 0.5
Decision Tree Accuracy: 0.575


### Task 5 Conclusion

In this experiment, KNN performance slightly decreased after applying feature scaling (from 0.55 to 0.50). This may be due to the nature of the randomly generated dataset, where scaling did not improve distance relationships significantly.

However, in general, KNN typically benefits from feature scaling because it relies on distance calculations, and scaling ensures all features contribute equally.

Decision Tree performance remained unchanged with scaling (0.575) because it is not based on distance but on feature splitting.

Therefore, while scaling is usually important for KNN, its impact can vary depending on the dataset.


## Task 6: Building a Pipeline

In this task, we create a machine learning pipeline to combine preprocessing and model training into one workflow.

We will:
- Build a pipeline using StandardScaler and KNN
- Train and test the pipeline
- Evaluate the model accuracy
- Understand why pipelines are useful in machine learning

In [55]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Prepare features and target
X = df.drop(columns=['heart_disease'])
y = df['heart_disease']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Build pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

# Train pipeline
pipeline.fit(X_train, y_train)

# Predict and evaluate
y_pred = pipeline.predict(X_test)
print("Pipeline KNN Accuracy:", accuracy_score(y_test, y_pred))

Pipeline KNN Accuracy: 0.475


### Task 6 Conclusion

The pipeline combines preprocessing and model training into one organized workflow.

Using a pipeline makes the code cleaner and ensures that scaling is applied consistently during both training and testing.

Pipelines are useful because they reduce the chance of mistakes, improve reproducibility, and make machine learning workflows easier to manage.

## Task 7: Feature Selection

In this task, we analyze feature importance using correlation and select the most relevant features.

We will:
- Compute the correlation matrix
- Identify features most related to the target variable
- Select important features based on a threshold
- Compare model performance using selected features

In [57]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd

# Prepare features and target
feature_cols = df.drop(columns=['heart_disease']).columns.tolist()
X = df.drop(columns=['heart_disease']).values
y = df['heart_disease'].values

# Scale features for correlation analysis
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Create DataFrame with scaled features
df_processed = pd.DataFrame(X_scaled, columns=feature_cols)
df_processed['heart_disease'] = y

# Compute correlation matrix
corr_matrix = df_processed.corr()

# Correlation with target
target_corr = corr_matrix['heart_disease'].drop('heart_disease')
print("Correlation with heart_disease:")
print(target_corr.sort_values(ascending=False).round(3))

# Select important features
threshold = 0.05
important_features = target_corr[abs(target_corr) > threshold]

print(f"\nFeatures with |correlation| > {threshold}:")
print(important_features.sort_values(ascending=False).round(3))

# Train model using selected features
top_features = important_features.index.tolist()
X_selected = df_processed[top_features].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)

pipe_selected = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

pipe_selected.fit(X_tr, y_tr)
y_pred = pipe_selected.predict(X_te)
sel_acc = accuracy_score(y_te, y_pred)

print(f"\nAccuracy with selected features: {sel_acc:.3f}")
print(f"Features used: {len(top_features)} out of {len(feature_cols)}")
print("Selected feature names:", top_features)

Correlation with heart_disease:
age               0.144
blood_pressure    0.105
city_Riyadh       0.079
city_Dammam       0.067
gender_encoded    0.014
cholesterol       0.002
bmi              -0.004
smoker_encoded   -0.025
city_Makkah      -0.061
city_Jeddah      -0.089
Name: heart_disease, dtype: float64

Features with |correlation| > 0.05:
age               0.144
blood_pressure    0.105
city_Riyadh       0.079
city_Dammam       0.067
city_Makkah      -0.061
city_Jeddah      -0.089
Name: heart_disease, dtype: float64

Accuracy with selected features: 0.550
Features used: 6 out of 10
Selected feature names: ['age', 'blood_pressure', 'city_Dammam', 'city_Jeddah', 'city_Makkah', 'city_Riyadh']


### Task 7 Conclusion

Feature selection using correlation analysis helps identify which features are most relevant to the target variable.

By selecting only important features, the model can become simpler and more efficient without significantly reducing accuracy.

In some cases, using fewer features can even improve performance by reducing noise and avoiding overfitting.

Therefore, feature selection is an important step in improving model efficiency and performance.

## Final Reflection

In this lab, we explored important data preprocessing techniques used in machine learning.

We learned how to handle missing values using imputation, convert categorical variables into numerical form using encoding, and scale features to improve model performance.

We also compared different models and observed how scaling affects distance-based algorithms like KNN, while having little impact on tree-based models.

Finally, we applied feature selection to identify the most relevant features and reduce dataset complexity while maintaining performance.

Overall, this lab helped build a strong understanding of preprocessing steps and their importance in building effective machine learning models.